# Multimodal Vision Examples

This notebook demonstrates how to use the MLX Omni Server with Vision-Language Models (VLMs) for multimodal tasks involving images and text.

In [1]:
from openai import OpenAI
import base64
from io import BytesIO
from PIL import Image
import requests

# Configure client to use local server
client = OpenAI(
    base_url="http://localhost:10240/v1",  # Point to local server
    api_key="not-needed"  # API key is not required for local server
)

## Example 1: Image Description with Remote URL

Ask the model to describe an image from a remote URL.

In [ ]:
response = client.chat.completions.create(
    model="mlx-community/GLM-4.6V-Flash-mxfp4",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What's in this image?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
                    },
                },
            ],
        }
    ],
    max_tokens=300,
)

print(response.choices[0].message.content)

## Example 2: Image Description with Base64 Encoding

Ask the model to describe an image that you encode as base64.

In [ ]:
# Load an image and encode it as base64
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content))

# Convert to base64
buffered = BytesIO()
image.save(buffered, format="JPEG")
img_str = base64.b64encode(buffered.getvalue()).decode()

response = client.chat.completions.create(
    model="mlx-community/GLM-4.6V-Flash-mxfp4",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What's in this image?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{img_str}"
                    },
                },
            ],
        }
    ],
    max_tokens=300,
)

print(response.choices[0].message.content)

## Example 3: Multiple Images in One Request

Ask the model to compare two images.

In [ ]:
response = client.chat.completions.create(
    model="mlx-community/GLM-4.6V-Flash-mxfp4",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Compare these two images and describe their differences:"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
                    },
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/440px-PNG_transparency_demonstration_1.png",
                    },
                },
            ],
        }
    ],
    max_tokens=500,
)

print(response.choices[0].message.content)

## Example 4: Streaming Response with Images

Get a streaming response for an image query.

In [ ]:
response = client.chat.completions.create(
    model="mlx-community/GLM-4.6V-Flash-mxfp4",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image in detail:"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
                    },
                },
            ],
        }
    ],
    stream=True,
    max_tokens=300,
)

for chunk in response:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)